# SVM 核函数对比

## 目标

理解 SVM 及其核函数的应用。

- 理解最大间隔分类器
- 对比不同核函数的效果
- 分析参数 C 和 gamma 的影响
- 可视化 SVM 的决策边界

## 1. 数据准备 - 生成不同类型的数据

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_circles, make_moons, make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import pandas as pd

np.random.seed(42)

# 生成不同类型的数据集
def generate_datasets():
    datasets = {}
    
    # 线性可分数据
    X_linear, y_linear = make_classification(
        n_samples=300, n_features=2, n_redundant=0, n_informative=2,
        n_clusters_per_class=1, random_state=42
    )
    datasets['linear'] = (X_linear, y_linear, '线性可分')
    
    # 环形数据
    X_circles, y_circles = make_circles(n_samples=300, factor=0.5, noise=0.1, random_state=42)
    datasets['circles'] = (X_circles, y_circles, '环形数据')
    
    # 月牙数据
    X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=42)
    datasets['moons'] = (X_moons, y_moons, '月牙数据')
    
    # 重叠数据
    X_overlap, y_overlap = make_classification(
        n_samples=300, n_features=2, n_redundant=0, n_informative=2,
        n_clusters_per_class=2, random_state=42, flip_y=0.15
    )
    datasets['overlap'] = (X_overlap, y_overlap, '重叠数据')
    
    return datasets

datasets = generate_datasets()

print("生成的数据集:")
for name, (X, y, desc) in datasets.items():
    print(f"  {name}: {X.shape} - {desc}")

In [ ]:
# 可视化数据集
plt.figure(figsize=(14, 4))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    plt.subplot(1, 4, i+1)
    plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], c='blue', alpha=0.6, label='类别 0')
    plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], c='red', alpha=0.6, label='类别 1')
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{desc}\n({name})')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 不同核函数的对比

In [ ]:
# 定义不同的核函数
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
kernel_names = {
    'linear': '线性核',
    'poly': '多项式核',
    'rbf': '高斯核 (RBF)',
    'sigmoid': 'Sigmoid 核'
}

print("SVM 核函数:")
print("=" * 40)
for k, name in kernel_names.items():
    print(f"  {k}: {name}")

In [ ]:
# 在环形数据上测试不同核函数
X_circles, y_circles, desc_circles = datasets['circles']

plt.figure(figsize=(14, 4))

for i, kernel in enumerate(kernels):
    plt.subplot(1, 4, i+1)
    
    # 训练 SVM
    if kernel == 'poly':
        svm = SVC(kernel=kernel, degree=3, random_state=42)
    else:
        svm = SVC(kernel=kernel, random_state=42)
    
    svm.fit(X_circles, y_circles)
    
    # 绘制决策边界
    h = 0.02
    x_min, x_max = X_circles[:, 0].min() - 1, X_circles[:, 0].max() + 1
    y_min, y_max = X_circles[:, 1].min() - 1, X_circles[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.Blues_r, alpha=0.3)
    plt.contourf(xx, yy, Z, levels=np.linspace(0, Z.max(), 7), cmap=plt.cm.Reds, alpha=0.3)
    plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
    
    plt.scatter(X_circles[y_circles == 0][:, 0], X_circles[y_circles == 0][:, 1], c='blue', alpha=0.6)
    plt.scatter(X_circles[y_circles == 1][:, 0], X_circles[y_circles == 1][:, 1], c='red', alpha=0.6)
    
    acc = svm.score(X_circles, y_circles)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{kernel_names[kernel]}\n准确率: {acc:.2f}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 在所有数据集上对比不同核函数

In [ ]:
# 在所有数据集上测试
results = []

for data_name, (X, y, desc) in datasets.items():
    for kernel in kernels:
        if kernel == 'poly':
            svm = SVC(kernel=kernel, degree=3, random_state=42)
        else:
            svm = SVC(kernel=kernel, random_state=42)
        
        svm.fit(X, y)
        acc = svm.score(X, y)
        
        results.append({
            '数据集': desc,
            '核函数': kernel_names[kernel],
            '准确率': acc
        })

# 显示结果
results_df = pd.DataFrame(results)
pivot_table = results_df.pivot(index='数据集', columns='核函数', values='准确率')
print("不同核函数在各数据集上的准确率:")
print(pivot_table.round(4))

# 可视化
plt.figure(figsize=(12, 6))
x = np.arange(len(pivot_table.index))
width = 0.2

for i, (kernel, name) in enumerate(zip(kernels, pivot_table.columns)):
    plt.bar(x + i*width, pivot_table[kernel_names[kernel]], width, label=name, alpha=0.8)

plt.xlabel('数据集')
plt.ylabel('准确率')
plt.title('不同核函数在各数据集上的性能对比')
plt.xticks(x + width*1.5, pivot_table.index)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

## 4. 参数 C 的影响（正则化强度）

In [ ]:
# 测试不同的 C 值
C_values = [0.01, 0.1, 1, 10, 100]

# 使用月牙数据
X_moons, y_moons, _ = datasets['moons']

plt.figure(figsize=(15, 4))

for i, C in enumerate(C_values):
    plt.subplot(1, 5, i+1)
    
    svm = SVC(kernel='rbf', C=C, random_state=42)
    svm.fit(X_moons, y_moons)
    
    # 绘制决策边界
    h = 0.02
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.Blues_r, alpha=0.3)
    plt.contourf(xx, yy, Z, levels=np.linspace(0, Z.max(), 7), cmap=plt.cm.Reds, alpha=0.3)
    plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
    
    # 标记支持向量
    plt.scatter(X_moons[svm.support_][:, 0], X_moons[svm.support_][:, 1],
                s=100, facecolors='none', edgecolors='yellow', linewidths=2)
    plt.scatter(X_moons[y_moons == 0][:, 0], X_moons[y_moons == 0][:, 1], c='blue', alpha=0.6)
    plt.scatter(X_moons[y_moons == 1][:, 0], X_moons[y_moons == 1][:, 1], c='red', alpha=0.6)
    
    acc = svm.score(X_moons, y_moons)
    n_support = len(svm.support_)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'C={C}\n准确率: {acc:.2f}\n支持向量: {n_support}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("参数 C 的说明:")
print("- C 小：允许更多错误，边界更平滑（欠拟合风险）")
print("- C 大：减少错误，边界更复杂（过拟合风险）")
print("- C 是正则化的倒数，C 越大，正则化越弱")

## 5. 参数 gamma 的影响（RBF 核宽度）

In [ ]:
# 测试不同的 gamma 值
gamma_values = [0.01, 0.1, 1, 10, 50]

plt.figure(figsize=(15, 4))

for i, gamma in enumerate(gamma_values):
    plt.subplot(1, 5, i+1)
    
    svm = SVC(kernel='rbf', gamma=gamma, random_state=42)
    svm.fit(X_moons, y_moons)
    
    # 绘制决策边界
    h = 0.02
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.Blues_r, alpha=0.3)
    plt.contourf(xx, yy, Z, levels=np.linspace(0, Z.max(), 7), cmap=plt.cm.Reds, alpha=0.3)
    plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
    
    # 标记支持向量
    plt.scatter(X_moons[svm.support_][:, 0], X_moons[svm.support_][:, 1],
                s=100, facecolors='none', edgecolors='yellow', linewidths=2)
    plt.scatter(X_moons[y_moons == 0][:, 0], X_moons[y_moons == 0][:, 1], c='blue', alpha=0.6)
    plt.scatter(X_moons[y_moons == 1][:, 0], X_moons[y_moons == 1][:, 1], c='red', alpha=0.6)
    
    acc = svm.score(X_moons, y_moons)
    n_support = len(svm.support_)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'γ={gamma}\n准确率: {acc:.2f}\n支持向量: {n_support}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("参数 gamma 的说明:")
print("- gamma 小：影响范围大，边界平滑（欠拟合风险）")
print("- gamma 大：影响范围小，边界复杂（过拟合风险）")
print("- gamma = 'scale': 默认，gamma = 1 / (n_features * X.var())")

## 6. C 和 gamma 的网格搜索

In [ ]:
# 使用网格搜索寻找最佳参数
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_moons, y_moons)

print("最佳参数:")
print(f"  C = {grid_search.best_params_['C']}")
print(f"  gamma = {grid_search.best_params_['gamma']}")
print(f"最佳交叉验证得分: {grid_search.best_score_:.4f}")
print(f"最佳模型支持向量数: {len(grid_search.best_estimator_.support_)}")

In [ ]:
# 可视化参数网格搜索结果
cv_results = grid_search.cv_results_

# 提取结果
C_values = param_grid['C']
gamma_values = param_grid['gamma']
scores = cv_results['mean_test_score'].reshape(len(C_values), len(gamma_values))

# 绘制热图
plt.figure(figsize=(10, 8))
plt.imshow(scores, interpolation='nearest', cmap=plt.cm.hot)
plt.xlabel('gamma')
plt.ylabel('C')
plt.colorbar(label='准确率')
plt.xticks(range(len(gamma_values)), gamma_values)
plt.yticks(range(len(C_values)), C_values)
plt.title('参数网格搜索结果（准确率）')

# 标记最佳参数
best_idx = np.unravel_index(np.argmax(scores), scores.shape)
plt.scatter([best_idx[1]], [best_idx[0]], s=200, c='yellow', marker='*',
           edgecolors='black', linewidths=2, label='最佳参数')
plt.legend()
plt.show()

print(f"\n最佳位置: C={C_values[best_idx[0]]}, gamma={gamma_values[best_idx[1]]}")

## 7. 多项式核的 degree 参数

In [ ]:
# 测试不同的多项式次数
degrees = [2, 3, 4, 5, 6]

# 使用环形数据
X_circles, y_circles, _ = datasets['circles']

plt.figure(figsize=(15, 4))

for i, degree in enumerate(degrees):
    plt.subplot(1, 5, i+1)
    
    svm = SVC(kernel='poly', degree=degree, random_state=42)
    svm.fit(X_circles, y_circles)
    
    # 绘制决策边界
    h = 0.02
    x_min, x_max = X_circles[:, 0].min() - 1, X_circles[:, 0].max() + 1
    y_min, y_max = X_circles[:, 1].min() - 1, X_circles[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.Blues_r, alpha=0.3)
    plt.contourf(xx, yy, Z, levels=np.linspace(0, Z.max(), 7), cmap=plt.cm.Reds, alpha=0.3)
    plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='black')
    
    # 标记支持向量
    plt.scatter(X_circles[svm.support_][:, 0], X_circles[svm.support_][:, 1],
                s=100, facecolors='none', edgecolors='yellow', linewidths=2)
    plt.scatter(X_circles[y_circles == 0][:, 0], X_circles[y_circles == 0][:, 1], c='blue', alpha=0.6)
    plt.scatter(X_circles[y_circles == 1][:, 0], X_circles[y_circles == 1][:, 1], c='red', alpha=0.6)
    
    acc = svm.score(X_circles, y_circles)
    n_support = len(svm.support_)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'degree={degree}\n准确率: {acc:.2f}\n支持向量: {n_support}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("多项式核的 degree 参数说明:")
print("- degree 小：边界简单，计算快")
print("- degree 大：边界复杂，计算慢")
print("- 对于环形数据，degree=2 或 3 通常就足够")

## 8. 真实数据集：乳腺癌分类

In [ ]:
# 加载真实数据集
cancer = load_breast_cancer()
X_real = cancer.data
y_real = cancer.target
feature_names = cancer.feature_names
target_names = cancer.target_names

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_real, y_real, test_size=0.3, random_state=42, stratify=y_real
)

# 标准化（SVM 对特征尺度敏感）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"数据集形状: {X_real.shape}")
print(f"类别名称: {target_names}")
print(f"类别分布: {np.bincount(y_real)}")
print(f"\n训练集: {X_train.shape}")
print(f"测试集: {X_test.shape}")

In [ ]:
# 在真实数据集上测试不同核函数
real_results = []

for kernel in kernels:
    if kernel == 'poly':
        svm_real = SVC(kernel=kernel, degree=3, random_state=42)
    else:
        svm_real = SVC(kernel=kernel, random_state=42)
    
    svm_real.fit(X_train_scaled, y_train)
    
    train_score = svm_real.score(X_train_scaled, y_train)
    test_score = svm_real.score(X_test_scaled, y_test)
    cv_scores = cross_val_score(svm_real, X_train_scaled, y_train, cv=5)
    
    real_results.append({
        '核函数': kernel_names[kernel],
        '训练集准确率': train_score,
        '测试集准确率': test_score,
        '交叉验证准确率': cv_scores.mean(),
        '交叉验证标准差': cv_scores.std(),
        '支持向量数': len(svm_real.support_)
    })

# 显示结果
real_results_df = pd.DataFrame(real_results)
print("乳腺癌数据集上的 SVM 性能:")
print(real_results_df.round(4).to_string(index=False))

## 9. 调优最佳模型

In [ ]:
# 对最佳核函数进行网格搜索
param_grid_real = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1, 10]
}

grid_search_real = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid_real,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_real.fit(X_train_scaled, y_train)

print("最佳参数:")
for param, value in grid_search_real.best_params_.items():
    print(f"  {param}: {value}")

print(f"\n最佳交叉验证得分: {grid_search_real.best_score_:.4f}")
print(f"测试集准确率: {grid_search_real.score(X_test_scaled, y_test):.4f}")
print(f"支持向量数: {len(grid_search_real.best_estimator_.support_)}")

## 10. 分类报告和混淆矩阵

In [ ]:
# 使用最佳模型进行预测
best_svm = grid_search_real.best_estimator_
y_pred_real = best_svm.predict(X_test_scaled)

# 分类报告
print("分类报告:")
print(classification_report(y_test, y_pred_real, target_names=target_names, digits=3))

# 混淆矩阵
cm_real = confusion_matrix(y_test, y_pred_real)

plt.figure(figsize=(8, 6))
plt.imshow(cm_real, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('混淆矩阵')
tick_marks = np.arange(len(target_names))
plt.xticks(tick_marks, target_names)
plt.yticks(tick_marks, target_names)
plt.xlabel('预测标签')
plt.ylabel('真实标签')

# 添加数值
for i in range(cm_real.shape[0]):
    for j in range(cm_real.shape[1]):
        plt.text(j, i, format(cm_real[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm_real[i, j] > cm_real.max() / 2 else "black")

plt.colorbar()
plt.tight_layout()
plt.show()

## 11. 支持向量的分析

In [ ]:
# 分析支持向量
print(f"支持向量总数: {len(best_svm.support_)}")
print(f"训练样本总数: {len(X_train_scaled)}")
print(f"支持向量比例: {len(best_svm.support_) / len(X_train_scaled):.2%}")

# 每个类别的支持向量
print("\n各类别支持向量:")
for i, class_name in enumerate(target_names):
    class_support = np.sum(y_train[best_svm.support_] == i)
    class_total = np.sum(y_train == i)
    print(f"  {class_name}: {class_support} / {class_total} ({class_support/class_total:.1%})")

## 12. SVM 与其他分类器的对比

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# 定义不同的分类器
classifiers = {
    '逻辑回归': LogisticRegression(random_state=42, max_iter=1000),
    '决策树': DecisionTreeClassifier(random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42)
}

# 训练和评估
comparison_results = []

for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    
    train_score = clf.score(X_train_scaled, y_train)
    test_score = clf.score(X_test_scaled, y_test)
    cv_scores = cross_val_score(clf, X_train_scaled, y_train, cv=5)
    
    comparison_results.append({
        '分类器': name,
        '训练集准确率': train_score,
        '测试集准确率': test_score,
        '交叉验证准确率': cv_scores.mean(),
        '交叉验证标准差': cv_scores.std()
    })

# 显示结果
comparison_df = pd.DataFrame(comparison_results)
print("不同分类器在乳腺癌数据集上的性能:")
print(comparison_df.round(4).to_string(index=False))

## 总结

### SVM 核函数对比

| 核函数 | 公式 | 特点 | 适用场景 |
|--------|------|------|----------|
| Linear | K(x,y) = xᵀy | 线性分类器 | 线性可分数据 |
| Polynomial | K(x,y) = (γxᵀy + r)^d | 多项式边界 | 具有特定几何结构的数据 |
| RBF | K(x,y) = exp(-γ||x-y||²) | 高斯径向基基函数 | 非线性、复杂边界 |
| Sigmoid | K(x,y) = tanh(γxᵀy + r) | 类似神经网络 | 特定应用场景 |

### 关键参数

1. **C（正则化参数）**:
   - C 小：强正则化，边界平滑，容忍错误
   - C 大：弱正则化，边界复杂，减少错误
   - 默认 C=1

2. **gamma（RBF 核参数）**:
   - gamma 小：影响范围大，边界平滑
   - gamma 大：影响范围小，边界复杂
   - 默认 gamma='scale'

3. **degree（多项式核参数）**:
   - 控制多项式的次数
   - 默认 degree=3

### SVM 的优势与局限

**优势**:
- 在高维空间中表现良好
- 在特征维度大于样本数时仍有效
- 使用核函数可处理非线性问题
- 决策边界由支持向量唯一确定

**局限**:
- 对大规模数据集训练慢
- 对噪声和异常值敏感
- 需要特征标准化
- 参数调优较复杂

### 实践建议

- 默认使用 RBF 核函数
- 始终进行特征标准化
- 使用网格搜索调优参数
- 对于线性可分数据，优先使用线性核
- 大数据集考虑使用线性核或其他算法